In [1]:
import numpy as np
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset


In [2]:
class SimplestModel(nn.Module):
    def __init__(
        self,
        inputs=4,
        bias=True,
        criterion=nn.MSELoss,
        optimizer=optim.SGD,
        lr=0.001,
        local_epochs=2,
    ):
        super().__init__()
        # Network architecture
        self.linear = nn.Linear(inputs, 1, bias=bias)
        self.criterion = criterion()
        self.optimizer = optimizer(self.parameters(), lr=lr)
        self.local_epochs = local_epochs

    def forward(self, x):
        pred = self.linear(x)
        return pred

    

In [3]:
# random datasets
num_samples = 2000
num_features = 4

# Generate random data
torch.manual_seed(0)  # For reproducibility
inputs = torch.randn(num_samples, num_features)
weights = torch.randn(num_features, 1)  # Random weights for our linear model
bias = torch.randn(1)  # Random bias for our linear model

# Create linear relationship y = Xw + b + noise
noise = torch.randn(num_samples, 1) * 0.1  # Noise to make the data a bit more realistic
labels = inputs @ weights + bias + noise

# Split the dataset into training, validation and test sets
train_size = int(0.6 * num_samples)
val_size = int(0.2 * num_samples)
test_size = num_samples - train_size - val_size

train_dataset = TensorDataset(inputs[:train_size], labels[:train_size])
val_dataset = TensorDataset(inputs[train_size : train_size + val_size], labels[train_size : train_size + val_size])
test_dataset = TensorDataset(inputs[train_size + val_size :], labels[train_size + val_size :])

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [4]:
# tests 
def concatenate_weights(weights_list):
    flattened_weights = []
    shapes = []

    for layer in weights_list:
        for weight_matrix in layer:
            # Flatten and store the weight matrix
            flattened_weights.append(weight_matrix.flatten())
            # Store the shape for reconstruction
            if len(weight_matrix.shape) > 0:
                shapes.append(weight_matrix.shape)
            else:
                shapes.append((1,))

    # Concatenate all flattened weights into a single vector
    concatenated_weights = np.concatenate(flattened_weights)

    return concatenated_weights, shapes

def deconcatenate_weights(flat_vector, shapes):
    reconstructed_weights = []
    idx = 0

    for shape in shapes:
        size = np.prod(shape)
        weight_matrix = flat_vector[idx:idx + size].reshape(shape)
        reconstructed_weights.append(weight_matrix)
        idx += size

    return reconstructed_weights


class NamedArray:
    def __init__(self, array, name=None):
        self.array = np.array(array)
        self.name = name

def split_weights(weights_list, n_splits, seed=42):
    # Flatten the matrix
    flat_matrix, shapes = concatenate_weights(weights_list)

    # Shuffle both using the same seed
    np.random.seed(seed)
    np.random.shuffle(flat_matrix)

    # Split the matrix into N parts and assign names
    splitted_parts = np.array_split(flat_matrix, n_splits)
    named_parts = [NamedArray(part, name=str(i)) for i, part in enumerate(splitted_parts)]

    return named_parts, shapes


def reconstruct_weights(splitted_named_matrices, shapes, seed=42):
    # Sort the parts based on their names
    sorted_parts = sorted(splitted_named_matrices, key=lambda x: int(x.name))
    
    # Concatenate the parts
    concatenated = np.concatenate([part.array for part in sorted_parts])

    # Shuffle the concatenated array to reconstruct the original order
    ordered_vector = np.arange(len(concatenated))
    np.random.seed(seed)
    np.random.shuffle(ordered_vector)

    # Reconstruct the original matrix
    reconstructed = np.empty_like(concatenated)
    for i, index in enumerate(ordered_vector):
        reconstructed[index] = concatenated[i]

    return deconcatenate_weights(reconstructed, shapes)


model = SimplestModel()
weights_torch = []
for param in model.parameters():
    weights_torch.append(param.data.numpy())

# example
print("Original Keras weights:\n", weights_torch)
w, shapes = split_weights(weights_torch, 3)
# print(w[0].array, w[1].array, w[2].array)
r = reconstruct_weights(w, shapes)
print("Reconstructed weights:\n", r)

Original Keras weights:
 [array([[0.07684439, 0.0257929 , 0.28234786, 0.48812044]], dtype=float32), array([-0.20136291], dtype=float32)]
Reconstructed weights:
 [array([0.07684439, 0.0257929 , 0.28234786, 0.48812044], dtype=float32), array([-0.20136291], dtype=float32)]


## Split model for iDLG

In [17]:
weight_list = []
for g in model.parameters():
    weight_list.append(g.data.numpy())

In [ ]:
def concatenate_weights(weights_list, n_splits=0):
    flattened_weights = []
    shapes = []

    for layer in weights_list:
        for weight_matrix in layer:
            # Flatten and store the weight matrix
            flattened_weights.append(weight_matrix.flatten())
            # Store the shape for reconstruction
            if len(weight_matrix.shape) > 0:
                shapes.append(weight_matrix.shape)
            else:
                shapes.append((1,))

    # Concatenate all flattened weights into a single vector
    concatenated_weights = np.concatenate(flattened_weights)
    
    # zero out parameters 
    if n_splits > 0:
        ...
        

    return concatenated_weights, shapes

def deconcatenate_weights(flat_vector, shapes):
    reconstructed_weights = []
    idx = 0

    for shape in shapes:
        size = np.prod(shape)
        weight_matrix = flat_vector[idx:idx + size].reshape(shape)
        reconstructed_weights.append(weight_matrix)
        idx += size

    return reconstructed_weights

w, s = concatenate_weights(weight_list)
r = deconcatenate_weights(w,s)
r

[array([0.07684439, 0.0257929 , 0.28234786, 0.48812044], dtype=float32),
 array([-0.20136291], dtype=float32)]

In [19]:
weight_list

[array([[0.07684439, 0.0257929 , 0.28234786, 0.48812044]], dtype=float32),
 array([-0.20136291], dtype=float32)]

In [20]:
w

array([ 0.07684439,  0.0257929 ,  0.28234786,  0.48812044, -0.20136291],
      dtype=float32)

### All inside the model


In [6]:
class SimplestModel(nn.Module):
    def __init__(
        self,
        inputs=4,
        bias=True,
        criterion=nn.MSELoss,
        optimizer=optim.SGD,
        lr=0.001,
        local_epochs=200,
    ):
        super().__init__()
        # Network architecture
        self.linear = nn.Linear(inputs, 1, bias=bias)
        self.criterion = criterion()
        self.optimizer = optimizer(self.parameters(), lr=lr)
        self.local_epochs = local_epochs
        self.shapes = None

    def forward(self, x):
        pred = self.linear(x)
        return pred

    def fit(self, train_loader):
        # Training loop
        for epoch in range(self.local_epochs):
            self.train()  # Set model to training mode
            total_loss = 0
            for batch, (x, y) in enumerate(train_loader):
                # Zero the gradients
                self.optimizer.zero_grad()

                # Perform forward pass
                outputs = self(x.float())  # Ensure input data type matches model expected type

                # Compute loss
                loss = self.criterion(outputs, y.float())  # Ensure label data type matches output

                # Backward pass and optimize
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()

            # Print average loss for the epoch
            print(f'Epoch {epoch+1}/{self.local_epochs}, Loss: {total_loss / (batch + 1)}')
            
    def evaluate(self, val_loader):
        self.eval()
        with torch.no_grad():
            test_loss = 0
            for x, y in val_loader:
                outputs = self(x.float())
                loss = self.criterion(outputs, y.float())
                test_loss += loss.item()
            print(f'Val Loss: {test_loss / len(val_loader)}')
    
    # def get_weights(self):
    #     return [param.data.numpy() for param in self.parameters()]
    def get_weights(self):
        weights = [param.data.numpy() for param in self.parameters()]
        self.shapes = [w.shape for w in weights]  # Store the original shapes
        return weights
    
    # def set_weights(self, weights):
    #     for i, param in enumerate(self.parameters()):
    #         param.data = torch.from_numpy(weights[i])
    def set_weights(self, weights):
        for i, param in enumerate(self.parameters()):
            # Ensure the new weights have the correct shape
            reshaped_weight = weights[i].reshape(self.shapes[i])
            param.data = torch.from_numpy(reshaped_weight)
    
    def split_weights(self, n_splits, seed=42):
        weights = self.get_weights()
        #splitted_matrices, self.shapes = split_weights(weights, n_splits, seed)
        splitted_matrices, _ = split_weights(weights, n_splits, seed)

        return splitted_matrices
    
    def reconstruct_weights(self, splitted_matrices, seed=42):
        weights = reconstruct_weights(splitted_matrices, self.shapes, seed)
        self.set_weights(weights)
        return weights

In [8]:
# create model
model = SimplestModel(inputs=num_features)


In [9]:
# train model
model.fit(train_loader)

Epoch 1/200, Loss: 1.1035212215624357
Epoch 2/200, Loss: 1.0190545163656537
Epoch 3/200, Loss: 0.9490953150548433
Epoch 4/200, Loss: 0.8741814776470787
Epoch 5/200, Loss: 0.8098913242942408
Epoch 6/200, Loss: 0.7503826210373327
Epoch 7/200, Loss: 0.6942177383523238
Epoch 8/200, Loss: 0.6435434316333971
Epoch 9/200, Loss: 0.5956704522434034
Epoch 10/200, Loss: 0.5511718235517803
Epoch 11/200, Loss: 0.5111623563264546
Epoch 12/200, Loss: 0.4734281960286592
Epoch 13/200, Loss: 0.43910881876945496
Epoch 14/200, Loss: 0.4061287061164254
Epoch 15/200, Loss: 0.3757274158691105
Epoch 16/200, Loss: 0.3476531019336299
Epoch 17/200, Loss: 0.32281357912640823
Epoch 18/200, Loss: 0.3007903600993909
Epoch 19/200, Loss: 0.27731806667227493
Epoch 20/200, Loss: 0.2574941790417621
Epoch 21/200, Loss: 0.23835216697893644
Epoch 22/200, Loss: 0.22163137950395284
Epoch 23/200, Loss: 0.2059839163955889
Epoch 24/200, Loss: 0.19091353918376722
Epoch 25/200, Loss: 0.1778181132517363
Epoch 26/200, Loss: 0.164601

In [10]:
# evaluate model
model.evaluate(val_loader)

Val Loss: 0.011922371573746204


In [11]:
# show the original weights
model.get_weights()

[array([[-0.35436144,  0.6347914 , -0.60469246,  0.30624884]],
       dtype=float32),
 array([0.8864555], dtype=float32)]

In [12]:
# randomly split the weights
N_SPLITS = 3
splitted_weights = model.split_weights(N_SPLITS)
print(splitted_weights[0].array, splitted_weights[1].array, splitted_weights[2].array)

[0.6347914 0.8864555] [-0.60469246 -0.35436144] [0.30624884]


In [13]:
# reconstruct the weights and check if they are the same as the original ones
model.reconstruct_weights(splitted_weights)

[array([[-0.35436144,  0.6347914 , -0.60469246,  0.30624884]],
       dtype=float32),
 array([0.8864555], dtype=float32)]

In [14]:
# check also if the loss is the same as before
model.evaluate(val_loader)

Val Loss: 0.011922371573746204
